# Notbook 001 · ¿Es indispensable sumar las entradas de una neurona?

**Objetivo:** cuestionar la integración aditiva convencional y probar una interacción multiplicativa bajo una comparación controlada. Es un **experimento exploratorio**, no una arquitectura original demostrada.

**Pregunta falsable:** ¿una unidad con interacciones de segundo orden mejora la capacidad de aprender ciertas reglas *con aproximadamente el mismo número de parámetros* que una MLP convencional, y a qué coste de ejecución?

**Hipótesis H1:** en problemas donde importan las interacciones entre variables (XOR, círculos), la variante multiplicativa alcanzará mayor precisión o aprenderá antes con un presupuesto similar; en tareas lineales podría no aportar ventajas.

**Condiciones de comparación:** misma partición de datos, pérdida, optimizador, cantidad de actualizaciones, semillas y dispositivo CPU. Mediremos parámetros, precisión fuera de muestra, robustez a extensión del dominio, tiempo de entrenamiento e inferencia. Esta comparación **no iguala FLOPs**, que deben medirse en fases posteriores.

**Fuentes de antecedentes:**
- Schmitt, *On the Complexity of Computing and Learning with Multiplicative Neural Networks*, Neural Computation, 2002. https://pubmed.ncbi.nlm.nih.gov/11802913/
- Fan y col., *On Expressivity and Trainability of Quadratic Networks*, IEEE TNNLS, 2025. https://pubmed.ncbi.nlm.nih.gov/37995168/
- Liu y col., *Dendritic Integration Inspired Artificial Neural Networks Capture Data Correlation*, NeurIPS 2024. https://proceedings.nips.cc/paper_files/paper/2024/hash/90b31ad371165eaac2dc6de8993fded7-Abstract-Conference.html

### Modelos

1. **Lineal:** `Linear(2,1)` (3 parámetros): control sin capa oculta.
2. **Aditiva:** `Linear(2,7) → SiLU → Linear(7,1)` (**29 parámetros**).
3. **Interactiva:** `SiLU(Wx+b+α⊙(Ux)⊙(Vx)) → Linear(3,1)` (**28 parámetros**).

Los modelos 2 y 3 tienen presupuestos similares. El lineal es solo un control inferior.


In [ ]:
import time, statistics, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibilidad práctica en CPU, sin exigir GPU de Colab.
torch.set_num_threads(1)
DEVICE = torch.device('cpu')
SEEDS = [11, 23, 37]  # Piloto; ampliar a 10 semillas en estudio definitivo.
STEPS = 250
N_TRAIN, N_TEST = 1024, 2048
LEARNING_RATE = 0.03
print('PyTorch:', torch.__version__, '| Dispositivo:', DEVICE)


In [ ]:
class Lineal(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)
    def forward(self, x):
        return self.fc(x).squeeze(-1)

class Aditiva(nn.Module):
    def __init__(self):
        super().__init__()
        self.entrada = nn.Linear(2, 7)
        self.salida = nn.Linear(7, 1)
    def forward(self, x):
        return self.salida(F.silu(self.entrada(x))).squeeze(-1)

class Interactiva(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = nn.Linear(2, 3)
        self.u = nn.Linear(2, 3, bias=False)
        self.v = nn.Linear(2, 3, bias=False)
        self.alpha = nn.Parameter(torch.full((3,), 0.1))
        self.salida = nn.Linear(3, 1)
    def forward(self, x):
        z = self.base(x) + self.alpha * self.u(x) * self.v(x)
        return self.salida(F.silu(z)).squeeze(-1)

MODELOS = {'Lineal': Lineal, 'Aditiva': Aditiva, 'Interactiva': Interactiva}
num_params = lambda m: sum(p.numel() for p in m.parameters())
print({nombre: num_params(ctor()) for nombre, ctor in MODELOS.items()})
assert num_params(Aditiva()) == 29
assert num_params(Interactiva()) == 28


### Tres problemas independientes

- **XOR de signos:** la respuesta depende de la combinación de los dos valores.
- **Círculos:** la respuesta depende de la distancia al origen.
- **Lineal:** no debería necesitar multiplicación entre entradas.

El test de evaluación utiliza ejemplos **nuevos**. Añadimos un test fuera del rango entrenado para obtener una primera señal de extrapolación, sin confundirlo con una garantía general.


In [ ]:
def crear_datos(tarea, n, seed, escala=1.0):
    g = torch.Generator().manual_seed(seed)
    x = (torch.rand((n, 2), generator=g) * 2 - 1) * escala
    if tarea == 'XOR':
        y = (x[:, 0] * x[:, 1] > 0).float()
    elif tarea == 'Circulos':
        # Región interior del círculo de radio 0.75
        y = (x.square().sum(dim=1) < 0.75**2).float()
    elif tarea == 'Lineal':
        y = (x[:, 0] + 0.6 * x[:, 1] > 0).float()
    else:
        raise ValueError(tarea)
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def accuracy(modelo, x, y):
    return ((modelo(x) >= 0).float() == y).float().mean().item()

def entrenar(modelo, x, y, pasos=STEPS):
    modelo.train()
    optim = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
    ini = time.perf_counter()
    for _ in range(pasos):
        optim.zero_grad(set_to_none=True)
        error = F.binary_cross_entropy_with_logits(modelo(x), y)
        error.backward()
        optim.step()
    return time.perf_counter() - ini

# Se usan exactamente los mismos datos para todos los modelos dentro de cada tarea/semilla.
filas=[]
for tarea in ['XOR', 'Circulos', 'Lineal']:
    for seed in SEEDS:
        train_x, train_y = crear_datos(tarea, N_TRAIN, 1000 + seed)
        test_x, test_y = crear_datos(tarea, N_TEST, 2000 + seed)
        extra_x, extra_y = crear_datos(tarea, N_TEST, 3000 + seed, escala=1.5)
        for nombre, constructor in MODELOS.items():
            torch.manual_seed(seed)
            random.seed(seed)
            np.random.seed(seed)
            modelo = constructor().to(DEVICE)
            segundos = entrenar(modelo, train_x, train_y)
            modelo.eval()
            filas.append({
                'tarea': tarea, 'semilla': seed, 'modelo': nombre,
                'parametros': num_params(modelo),
                'precision_train': accuracy(modelo, train_x, train_y),
                'precision_test': accuracy(modelo, test_x, test_y),
                'precision_extrapolacion': accuracy(modelo, extra_x, extra_y),
                'entrenamiento_s': segundos,
            })

resultados = pd.DataFrame(filas)
resumen = resultados.groupby(['tarea','modelo'], as_index=False).agg(
    parametros=('parametros','first'),
    precision_test_media=('precision_test','mean'),
    precision_test_desv=('precision_test','std'),
    extrapolacion_media=('precision_extrapolacion','mean'),
    entrenamiento_s_media=('entrenamiento_s','mean'),
)
print('Resultados en datos no vistos (3 semillas):')
print(resumen.round(4).to_string(index=False))


In [ ]:
# Comparar inferencia en el MISMO CPU. Batch grande reduce el ruido de Python.
@torch.no_grad()
def medir_inferencia(modelo, n=4096, repeticiones=5, iteraciones=50):
    modelo.eval()
    x = torch.randn(n, 2, device=DEVICE)
    for _ in range(10):
        modelo(x)
    tiempos=[]
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        for _ in range(iteraciones):
            modelo(x)
        tiempos.append((time.perf_counter() - t0) * 1000 / iteraciones)
    return statistics.median(tiempos)

tabla_velocidad = pd.DataFrame([
    {'modelo': nombre, 'parametros': num_params(ctor()),
     'ms_por_lote_4096': medir_inferencia(ctor())}
    for nombre, ctor in MODELOS.items()
])
print(tabla_velocidad.round(4).to_string(index=False))
print('Advertencia: estas cifras dependen del hardware y no equivalen a un benchmark de FLOPs.')


In [ ]:
# Visualización del desempeño en test (sin utilizar el conjunto de prueba para entrenar).
for tarea in ['XOR', 'Circulos', 'Lineal']:
    sub = resumen[resumen['tarea'] == tarea]
    plt.figure(figsize=(5.5, 3.3))
    plt.bar(sub['modelo'], sub['precision_test_media'], yerr=sub['precision_test_desv'], capsize=5)
    plt.ylim(0, 1.05)
    plt.ylabel('Precisión en datos no vistos')
    plt.title(f'Tarea: {tarea} | {len(SEEDS)} semillas')
    plt.tight_layout()
    plt.show()


### Regla de decisión

No declararemos que la candidata es superior por ganar en XOR o círculos.

**Para continuar investigándola**, debe ganar de forma reproducible en al menos una tarea no trivial, sin empeorar de manera injustificada el tiempo o la robustez; después se evaluará con más semillas, más tareas y **cómputo comparable** (parámetros parecidos no implican igual coste). Si no muestra una ventaja útil, eliminaremos la interacción multiplicativa y probaremos otra hipótesis.

**Siguientes ablaciones propuestas:** quitar el sesgo, eliminar la activación SiLU, variar el rango y la distribución de las entradas, controlar FLOPs reales y evaluar estabilidad con 10 semillas. Ninguna configuración se considerará una neurona nueva sin revisar antecedentes.
